<a href="https://colab.research.google.com/github/ahmedalsufyan/IBM-Applied-Data-Science-Capstone/blob/main/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

# 1. Page Configuration
st.set_page_config(
    page_title="SpaceX Launch Analysis",
    page_icon="🚀",
    layout="wide"
)

# 2. Title & Overview
st.title("🚀 SpaceX Falcon 9 Launch Dashboard")
st.markdown("Interactive analysis of SpaceX launch sites, payload mass, and landing outcomes.")

# 3. Load Dataset
@st.cache_data
def load_data():
    dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
    data = pd.read_csv(dataset_url)
    return data

df = load_data()

# 4. Sidebar Controls
st.sidebar.header("Filter Options")
sites = ["ALL Sites"] + list(df['LaunchSite'].unique())
selected_site = st.sidebar.selectbox("Select Launch Site:", sites)

# Filter dataframe based on selection
if selected_site == "ALL Sites":
    filtered_df = df
else:
    filtered_df = df[df['LaunchSite'] == selected_site]

# 5. Key Metrics Summary
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Launches", len(filtered_df))
col2.metric("Successful Landings", int(filtered_df['Class'].sum()))
col3.metric("Success Rate", f"{filtered_df['Class'].mean() * 100:.1f}%")
col4.metric("Avg Payload (kg)", f"{filtered_df['PayloadMass'].mean():,.0f}")

st.markdown("---")

# 6. Interactive Charts
col_chart1, col_chart2 = st.columns(2)

with col_chart1:
    st.subheader("Success vs Failure Ratio")
    if selected_site == "ALL Sites":
        fig_pie = px.pie(
            df, values='Class', names='LaunchSite',
            title='Total Successful Landings by Site'
        )
    else:
        site_outcomes = filtered_df['Class'].value_counts().reset_index()
        site_outcomes.columns = ['Class', 'Count']
        site_outcomes['Outcome'] = site_outcomes['Class'].map({1: 'Success', 0: 'Failure'})
        fig_pie = px.pie(
            site_outcomes, values='Count', names='Outcome',
            title=f'Landing Outcomes for {selected_site}',
            color='Outcome', color_discrete_map={'Success': '#00CC96', 'Failure': '#EF553B'}
        )
    st.plotly_chart(fig_pie, use_container_width=True)

with col_chart2:
    st.subheader("Payload Mass vs. Flight Number")
    fig_scatter = px.scatter(
        filtered_df, x="FlightNumber", y="PayloadMass",
        color="Class", symbol="Class",
        labels={"Class": "Landing Outcome (1=Success)"},
        title="Payload Mass relative to Flight Number",
        color_continuous_scale=["#EF553B", "#00CC96"]
    )
    st.plotly_chart(fig_scatter, use_container_width=True)

# 7. Raw Data Table Preview
with st.expander("📄 View Raw Dataset Preview"):
    st.dataframe(filtered_df)

Overwriting app.py


In [8]:
!pip install pyngrok streamlit plotly -q

import time
import subprocess
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3HMBTnW87JbLQdcmAvHDSfzwJnq_59EstgggdxvwX9c19fZqv"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# إغلاق أي جلسات سابقة
ngrok.kill()

# تشغيل السيرفر
subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(4)

# فتح النفق
public_url = ngrok.connect(8501)

print("\n" + "="*50)
print("🚀 Live SpaceX Streamlit App URL:")
print(public_url)
print("="*50)


🚀 Live SpaceX Streamlit App URL:
NgrokTunnel: "https://grinch-basics-unjustly.ngrok-free.dev" -> "http://localhost:8501"
